# Phase 2 — Normalization (Business Entity Resolution, Amazon ML Challenge 2026)

Script-agnostic text normalization, built and verified against the real
noisy examples from `reports/eda.md` and the handoff spec.

**Design constraints (from `PLAN.md` / `reports/eda.md`):**
- Handle Latin (incl. French-accented), Devanagari, Tamil, Telugu, Kannada,
  Bengali, Gujarati, Malayalam, Oriya, Gurmukhi — all confirmed present.
- Never decompose/strip Indic combining marks (categories `Mn`/`Mc` — their
  vowel signs). A naive `[^\w\s]` stripper does this by accident (measured
  bug, `reports/eda.md` §7) — fixed here by stripping on Unicode *category*
  (`P*`/`S*`/`C*` removed, `L*`/`N*`/`M*` kept), never on `\w`.
- No per-country hardcoded rule tables. Legal-suffix/stopword removal is
  *derived* from per-country token document-frequency at runtime — this is
  what makes France's `sarl`/`sas`/`eurl` fall out automatically in the same
  band as US `llc`/`inc`, with zero French-specific code.
- A small hand-written supplementary suffix list exists only behind a
  feature flag, for Phase 7 to ablate.
- Does **not** transliterate Indic scripts to Latin. Per `reports/eda.md`
  §5/6, the ADDRESS channel — not transliteration — is what recovers
  transliterated-name matches (98–99% via address vs 0.2–1.0% via name
  alone), because digits and much of the address text survive script
  differences untouched. Cross-script matching is Phase 3/4's job, not
  this module's.


## Imports and optional fast backend

`regex` supports `\p{Category}` natively in one compiled pass; stdlib `re` does not, so we fall back to a per-character `unicodedata.category()` scan when `regex` isn't installed. Behaviour is identical either way — this only affects throughput, never correctness.

In [1]:
from __future__ import annotations

import re
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, Iterable, List, Optional, Set, Tuple

try:
    import regex as _re_engine  # type: ignore
    _HAS_REGEX_MODULE = True
    _STRIP_RE = _re_engine.compile(r"[\p{P}\p{S}\p{C}]+")
except ImportError:
    _HAS_REGEX_MODULE = False
    _STRIP_RE = None

print("Fast `regex` backend available:", _HAS_REGEX_MODULE)


Fast `regex` backend available: False


## Category-based punctuation/symbol stripping

This is the direct fix for the `[^\w\s]`-shatters-Indic bug: strip by Unicode *category*, keeping letters (`L*`), numbers (`N*`), and marks (`M*` — includes Indic vowel signs) rather than relying on `\w`.

In [2]:
_KEEP_CATEGORY_PREFIXES = ("L", "N", "M")


def _strip_by_category_stdlib(s: str) -> str:
    """Fallback punctuation/symbol stripper using unicodedata.category().
    Equivalent to the `regex` \\p{P}\\p{S}\\p{C} pass, character by
    character. Correctly reports Devanagari vowel signs as 'Mn'/'Mc',
    which we explicitly keep."""
    out = []
    prev_was_space = False
    for ch in s:
        cat = unicodedata.category(ch)
        if cat[0] in _KEEP_CATEGORY_PREFIXES:
            out.append(ch)
            prev_was_space = False
        else:
            if not prev_was_space:
                out.append(" ")
                prev_was_space = True
    return "".join(out)


def strip_punct_symbols_keep_marks(s: str) -> str:
    """Strip P*/S*/C* Unicode categories, keep L*/N*/M*. Script-agnostic."""
    if _HAS_REGEX_MODULE:
        return _STRIP_RE.sub(" ", s)
    return _strip_by_category_stdlib(s)


# quick sanity check: the exact bug from reports/eda.md Sec.7
_bad_example = "\u092a\u094d\u0930\u093e\u0907\u0935\u0947\u091f"  # "praivet" in Devanagari
print("Before:", repr(_bad_example))
print("After :", repr(strip_punct_symbols_keep_marks(_bad_example)))
print("(combining marks preserved -- not shattered into loose consonants)")


Before: 'प्राइवेट'
After : 'प्राइवेट'
(combining marks preserved -- not shattered into loose consonants)


## Script-run splitting

Only used to decide *which* runs get Latin-style diacritic stripping (NFKD + drop combining marks). Indic runs are left in their NFKC-composed form untouched — decomposing them would strip the same vowel signs we just fixed the bug for.

In [3]:
_INDIC_RANGES: Tuple[Tuple[str, int, int], ...] = (
    ("devanagari", 0x0900, 0x097F),
    ("bengali", 0x0980, 0x09FF),
    ("gurmukhi", 0x0A00, 0x0A7F),
    ("gujarati", 0x0A80, 0x0AFF),
    ("oriya", 0x0B00, 0x0B7F),
    ("tamil", 0x0B80, 0x0BFF),
    ("telugu", 0x0C00, 0x0C7F),
    ("kannada", 0x0C80, 0x0CFF),
    ("malayalam", 0x0D00, 0x0D7F),
)


def _char_class(ch: str) -> str:
    """'indic' (any of the 9 scripts) or 'other' (Latin incl. accented,
    digits, punctuation, whitespace, or any non-Indic script -- French,
    Cyrillic, etc. all fall here and get the same diacritic-stripping
    treatment, a no-op for scripts with nothing to strip)."""
    cp = ord(ch)
    for _name, lo, hi in _INDIC_RANGES:
        if lo <= cp <= hi:
            return "indic"
    return "other"


def _split_runs(s: str) -> List[Tuple[str, str]]:
    """Group consecutive same-class characters into (class, text) runs."""
    if not s:
        return []
    runs: List[Tuple[str, str]] = []
    cur_class = _char_class(s[0])
    cur_chars = [s[0]]
    for ch in s[1:]:
        cls = _char_class(ch)
        if cls == cur_class:
            cur_chars.append(ch)
        else:
            runs.append((cur_class, "".join(cur_chars)))
            cur_class, cur_chars = cls, [ch]
    runs.append((cur_class, "".join(cur_chars)))
    return runs


def _strip_diacritics_latin(s: str) -> str:
    """NFKD-decompose and drop combining marks. Only ever called on
    'other'-class runs (Latin/accented-Latin/etc), never on Indic runs."""
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


print(_split_runs("\u0936\u094d\u0930\u0940 Traders"))


[('indic', 'श्री'), ('other', ' Traders')]


## Core `clean_text()` pipeline

1. NFKC normalize (safe for Indic -- keeps composed vowel-sign forms intact)
2. Split into indic-script vs other-script runs
3. Strip Latin-style diacritics only on 'other' runs
4. Rejoin, strip punctuation/symbols by Unicode category
5. Casefold + collapse whitespace

In [4]:
def clean_text(raw: str) -> str:
    if not isinstance(raw, str) or not raw.strip():
        return ""
    s = unicodedata.normalize("NFKC", raw)
    runs = _split_runs(s)
    pieces = []
    for cls, text in runs:
        if cls == "other":
            pieces.append(_strip_diacritics_latin(text))
        else:
            pieces.append(text)  # indic run, untouched
    joined = "".join(pieces)
    joined = strip_punct_symbols_keep_marks(joined)
    joined = joined.casefold()
    joined = re.sub(r"\s+", " ", joined).strip()
    return joined


def tokenize(clean: str) -> List[str]:
    """Whitespace tokenization over an already-cleaned string."""
    return clean.split() if clean else []


for example in ["Payne \u00c9nterprises", "PAYNE-ENRTPRMISES",
                "Établissements Dëleves EURL", "63 R. DE DIEPPE"]:
    print(f"{example!r:35s} -> {clean_text(example)!r}")


'Payne Énterprises'                 -> 'payne enterprises'
'PAYNE-ENRTPRMISES'                 -> 'payne enrtprmises'
'Établissements Dëleves EURL'       -> 'etablissements deleves eurl'
'63 R. DE DIEPPE'                   -> '63 r de dieppe'


## Numeric / address-key extraction

Per `reports/eda.md` §5, digit runs are the strongest script-invariant signal (they survive transliteration untouched), so they're a first-class output, extracted separately from the general tokenizer.

**Documented limitation (not a bug):** `"48"` vs `"4-8"` produce disjoint key sets under *any* exact-key canonicalization, including this one — it's already inside the measured 0.12% lexically-unreachable ceiling.

In [5]:
_CODE_CHUNK_RE = re.compile(r"[A-Za-z0-9]+(?:-[A-Za-z0-9]+)*")


def extract_numeric_keys(raw: str) -> Set[str]:
    """'AF-0684' -> {'af684'}; '1056-1060' -> {'1056','1060'} (range);
    '1056c' -> {'1056c'}."""
    if not isinstance(raw, str) or not raw.strip():
        return set()
    keys: Set[str] = set()
    for m in _CODE_CHUNK_RE.finditer(raw):
        chunk = m.group(0)
        if not any(c.isdigit() for c in chunk):
            continue
        parts = chunk.lower().split("-")
        if len(parts) > 1 and all(p.isdigit() for p in parts):
            for p in parts:
                keys.add(p.lstrip("0") or "0")
        else:
            joined = "".join(
                p.lstrip("0") if p.isdigit() and p.lstrip("0") else (p if not p.isdigit() else "0")
                for p in parts
            )
            if joined:
                keys.add(joined)
    return keys


def char_ngrams(clean: str, n: int = 3) -> List[str]:
    """Character n-grams over the cleaned (space-collapsed) string."""
    s = clean.replace(" ", "")
    if len(s) < n:
        return [s] if s else []
    return [s[i : i + n] for i in range(len(s) - n + 1)]


print("AF-0684    ->", extract_numeric_keys("AF-0684"))
print("1056-1060  ->", extract_numeric_keys("1056-1060"))
print("48 vs 4-8  ->", extract_numeric_keys("48"), "vs", extract_numeric_keys("4-8"),
      "(disjoint, as expected)")


AF-0684    -> {'af684'}
1056-1060  -> {'1060', '1056'}
48 vs 4-8  -> {'48'} vs {'4', '8'} (disjoint, as expected)


## Supplementary legal-suffix list (feature-flagged only)

**Correction applied this round:** the list previously had only abbreviations (`ltd`, `pvt`). Per `reports/eda.md` §4 the *full-word* forms are actually the higher-DF India tokens (`limited` 59.1% vs `ltd` 16.5%; `private` 48.9% vs `pvt` 13.8%) — added so Phase 7's ablation is representative, not accidentally testing only the smaller half of the signal. This list is never the primary mechanism — that's the `DocumentFrequencyModel` below.

In [6]:
_SUPPLEMENTARY_LEGAL_SUFFIXES: Dict[str, str] = {
    # India -- abbreviations
    "ltd": "limited", "pvt": "private", "co": "company", "llp": "llp",
    # India -- full words (added: these are the higher-DF forms, per EDA)
    "limited": "limited", "private": "private", "company": "company",
    # US
    "llc": "llc", "inc": "incorporated", "corp": "corporation",
    "pc": "pc", "l": "l", "p": "p",
    "corporation": "corporation", "incorporated": "incorporated",
    # France -- abbreviations
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl",
    "sci": "sci", "sa": "sa",
    # France -- full/expanded forms seen in French company law
    "societe": "societe",
}


def apply_supplementary_suffixes(tokens: List[str]) -> List[str]:
    return [_SUPPLEMENTARY_LEGAL_SUFFIXES.get(t, t) for t in tokens]


## Corpus-derived document-frequency model

**This**, not the hand-written list above, is the primary stopword/legal-suffix mechanism — the reason the pipeline generalizes to France (unseen at train time) with zero country-specific code. Confirmed empirically: France `sarl` 28.3% / `sas` 20.1% land in the same DF band as US `llc` 26.9% / `inc` 18.0%; the same mechanism also catches French function words (`de`, `du`, `des`) that an English stoplist would miss.

In [7]:
@dataclass
class DocumentFrequencyModel:
    """Streaming per-partition (e.g. per-country) token document frequency."""

    doc_count: Dict[str, int] = field(default_factory=lambda: defaultdict(int))
    token_doc_count: Dict[str, Counter] = field(default_factory=lambda: defaultdict(Counter))

    def update(self, partition: str, tokens: Iterable[str]) -> None:
        """Feed one record's (partition, unique-token-set). Streams --
        no need to hold the corpus in RAM."""
        uniq = set(tokens)
        if not uniq:
            return
        self.doc_count[partition] += 1
        c = self.token_doc_count[partition]
        for t in uniq:
            c[t] += 1

    def df_fraction(self, partition: str, token: str) -> float:
        n = self.doc_count.get(partition, 0)
        if n == 0:
            return 0.0
        return self.token_doc_count[partition].get(token, 0) / n

    def build_stoplists(self, min_df_fraction: float = 0.02) -> Dict[str, Set[str]]:
        """Per-partition stoplist: tokens whose DF exceeds min_df_fraction.
        A threshold, not a fixed top-K, so it scales to however many
        distinct legal-suffix/function-word tokens a country's corpus has."""
        stoplists: Dict[str, Set[str]] = {}
        for partition, n in self.doc_count.items():
            if n == 0:
                stoplists[partition] = set()
                continue
            thresh = n * min_df_fraction
            stoplists[partition] = {
                tok for tok, cnt in self.token_doc_count[partition].items() if cnt >= thresh
            }
        return stoplists

    def filter_high_df(self, partition: str, tokens: List[str], stoplists: Dict[str, Set[str]]) -> List[str]:
        stop = stoplists.get(partition, set())
        kept = [t for t in tokens if t not in stop]
        return kept if kept else list(tokens)


# demo: France sarl/sas should land in the same high-DF band as US llc/inc,
# derived purely from counts -- no country-specific code anywhere above.
_dfm_demo = DocumentFrequencyModel()
for i in range(283):
    _dfm_demo.update("france", tokenize(f"company{i} sarl"))
for i in range(201):
    _dfm_demo.update("france", tokenize(f"firm{i} sas"))
for i in range(269):
    _dfm_demo.update("us", tokenize(f"company{i} llc"))
_sl_demo = _dfm_demo.build_stoplists(min_df_fraction=0.15)
print("France stoplist:", _sl_demo["france"])
print("US stoplist:", _sl_demo["us"])
print("core_tokens('company7 sarl'):",
      _dfm_demo.filter_high_df("france", tokenize("company7 sarl"), _sl_demo))


France stoplist: {'sarl', 'sas'}
US stoplist: {'llc'}
core_tokens('company7 sarl'): ['company7']


## Per-record normalization: the public entry point

In [8]:
@dataclass
class NormalizedRecord:
    clean: str
    tokens: List[str]
    core_tokens: List[str]
    joined: str
    char_ngrams: List[str]
    numeric_keys: Set[str] = field(default_factory=set)


def normalize_record(
    raw_name: str,
    raw_address: str = "",
    *,
    partition: Optional[str] = None,
    dfm: Optional[DocumentFrequencyModel] = None,
    stoplists: Optional[Dict[str, Set[str]]] = None,
    use_supplementary_suffixes: bool = False,
    ngram_n: int = 3,
) -> NormalizedRecord:
    """Normalize one (name, address) pair into the five required outputs.
    core_tokens needs a fitted DocumentFrequencyModel + stoplists for the
    record's partition (country); without one it falls back to tokens
    unchanged (useful for ad-hoc calls with no corpus to fit against)."""
    clean = clean_text(raw_name)
    tokens = tokenize(clean)

    if use_supplementary_suffixes:
        tokens = apply_supplementary_suffixes(tokens)

    if dfm is not None and stoplists is not None and partition is not None:
        core_tokens = dfm.filter_high_df(partition, tokens, stoplists)
    else:
        core_tokens = list(tokens)

    joined = "".join(tokens)
    ngrams = char_ngrams(clean, n=ngram_n)
    numeric_keys = extract_numeric_keys(raw_address) | extract_numeric_keys(raw_name)

    return NormalizedRecord(
        clean=clean, tokens=tokens, core_tokens=core_tokens,
        joined=joined, char_ngrams=ngrams, numeric_keys=numeric_keys,
    )


_r = normalize_record("Thermal & Fils SASU", "20 Rue Parmentier")
print(_r)


NormalizedRecord(clean='thermal fils sasu', tokens=['thermal', 'fils', 'sasu'], core_tokens=['thermal', 'fils', 'sasu'], joined='thermalfilssasu', char_ngrams=['the', 'her', 'erm', 'rma', 'mal', 'alf', 'lfi', 'fil', 'ils', 'lss', 'ssa', 'sas', 'asu'], numeric_keys={'20'})


## Throughput measurement helper

**Not** the real cmslab number — this sandbox's CPU is unknown-spec and not the 48-core Xeon Gold 6240R. Re-measure on the actual server before trusting it; must sustain ≥100k records/sec/core per `PLAN.md`.

In [9]:
def measure_throughput(samples: List[str], iterations: int = 1) -> float:
    """Returns records/sec for clean_text() over `samples`."""
    t0 = time.perf_counter()
    for _ in range(iterations):
        for s in samples:
            clean_text(s)
    elapsed = time.perf_counter() - t0
    total = len(samples) * iterations
    return total / elapsed if elapsed > 0 else float("inf")


_sample = ["Guru Infra Private Limited", "RG Services L.L.C.",
           "Donet Partnership L.L.C."] * 500
_rate = measure_throughput(_sample, iterations=3)
print(f"THIS SANDBOX ONLY: {_rate:,.0f} records/sec -- re-measure on cmslab")


THIS SANDBOX ONLY: 78,875 records/sec -- re-measure on cmslab


## Tests

Same 19 checks as `tests/test_normalize.py`, run inline against the exact
examples from the handoff spec and `reports/eda.md`.

In [10]:
FAILURES = []

def check(name, cond, detail=""):
    status = "PASS" if cond else "FAIL"
    print(f"[{status}] {name}" + (f"  -- {detail}" if detail and not cond else ""))
    if not cond:
        FAILURES.append(name)

# 1. The exact bug from reports/eda.md Sec.7
devanagari_private = "\u092a\u094d\u0930\u093e\u0907\u0935\u0947\u091f"
out = clean_text(devanagari_private)
check("Indic combining marks preserved (not shattered)",
      out == devanagari_private.casefold() or len(out.replace(" ", "")) == len(devanagari_private),
      detail=f"got {out!r}")

# 2. Mixed script
mixed = "\u0936\u094d\u0930\u0940 Traders"
out = clean_text(mixed)
check("Mixed Devanagari+Latin: both segments present",
      "traders" in out and any(0x0900 <= ord(c) <= 0x097F for c in out), detail=f"got {out!r}")

# 3. Transliteration-noise pair
a = clean_text("Payne \u00c9nterprises")
b = clean_text("PAYNE-ENRTPRMISES")
check("Payne Énterprises -> diacritic stripped + lowercased", a == "payne enterprises", detail=repr(a))
check("PAYNE-ENRTPRMISES -> hyphen becomes separator, lowercased", b == "payne enrtprmises", detail=repr(b))

# 4. Empty inputs
rec = normalize_record("Some Company", "")
check("Empty address -> no crash", isinstance(rec.numeric_keys, set))
rec2 = normalize_record("", "")
check("Empty name+address -> clean is empty string", rec2.clean == "", detail=repr(rec2.clean))

# 5. French examples
fr1 = clean_text("Thermal & Fils SASU")
check("French '&' stripped, not swallowed", fr1 == "thermal fils sasu", detail=repr(fr1))
fr2 = clean_text("20 Rue Parmentier")
check("French address token", fr2 == "20 rue parmentier", detail=repr(fr2))
fr3 = clean_text("63 R. DE DIEPPE")
check("French 'R.' -> period stripped, not glued", fr3 == "63 r de dieppe", detail=repr(fr3))
fr4 = clean_text("Établissements Dëleves EURL")
check("French accents stripped (é->e, ë->e)", fr4 == "etablissements deleves eurl", detail=repr(fr4))

# 6. Real hard examples from EDA -- must not crash
hard_examples = [
    ("Guru Infra Private Limited", "House No. 671, C/O Memon Afaque, Sobani Manzil, Gawalipura, Sadar, Nagpur, Maharashtra"),
    ("Good Industries Private Limited", "48, 1St Floor, New Modella Co-Op Premises Soc Ltd Padwal Ngr, Panchpakhadi, Wagle Indl. Es, Tate, Thane, Maharashtra"),
    ("RG Services L.L.C.", "231 Alexander Boulevard, Clarksville, TN"),
    ("Donet Partnership L.L.C.", "702 Marian Court, Merced, CA"),
]
crashed = False
for name, addr in hard_examples:
    try:
        r = normalize_record(name, addr)
        assert r.clean and r.tokens
    except Exception as e:
        crashed = True
        print(f"  CRASH on {name!r}: {e}")
check("Real EDA hard-example records normalize without crashing", not crashed)

# 7. Numeric keys
k1 = extract_numeric_keys("AF-0684")
check("'AF-0684' -> {'af684'}", k1 == {"af684"}, detail=str(k1))
k2 = extract_numeric_keys("1056-1060")
check("'1056-1060' -> {'1056','1060'}", k2 == {"1056", "1060"}, detail=str(k2))
k3a, k3b = extract_numeric_keys("48, 1St Floor"), extract_numeric_keys("4-8, THANE")
check("Documented limitation: '48' vs '4-8' disjoint (expected)", k3a.isdisjoint(k3b), detail=f"{k3a} vs {k3b}")

# 8. DF stoplist (fresh model, distinct names so only the suffix is high-DF)
dfm = DocumentFrequencyModel()
for i in range(283):
    dfm.update("france", tokenize(f"company{i} sarl"))
for i in range(201):
    dfm.update("france", tokenize(f"firm{i} sas"))
for i in range(65):
    dfm.update("france", tokenize(f"maison{i} eurl"))
for i in range(269):
    dfm.update("us", tokenize(f"company{i} llc"))
for i in range(180):
    dfm.update("us", tokenize(f"firm{i} inc"))
stoplists = dfm.build_stoplists(min_df_fraction=0.15)
check("France 'sarl' derived into stoplist purely from DF", "sarl" in stoplists["france"], detail=str(stoplists["france"]))
check("US 'llc' derived into stoplist purely from DF", "llc" in stoplists["us"], detail=str(stoplists["us"]))
check("Partitions don't leak (France stoplist has no 'llc')", "llc" not in stoplists["france"])
core = dfm.filter_high_df("france", tokenize("company7 sarl"), stoplists)
check("core_tokens drops high-DF 'sarl', keeps 'company7'", core == ["company7"], detail=str(core))

# 9. No hardcoded country-name branching (grep-level self-check, computed
#    over the module-definition cells' own source text -- injected as
#    `_country_literal_findings` since inspect.getsource() can't recover
#    source for functions exec()'d from a string inside this notebook)
check("No hardcoded country-name branching", not _country_literal_findings,
      detail=str(_country_literal_findings))

print(f"\n{len(FAILURES)} failing check(s)." if FAILURES else "\nAll checks passed.")


[PASS] Indic combining marks preserved (not shattered)
[PASS] Mixed Devanagari+Latin: both segments present
[PASS] Payne Énterprises -> diacritic stripped + lowercased
[PASS] PAYNE-ENRTPRMISES -> hyphen becomes separator, lowercased
[PASS] Empty address -> no crash
[PASS] Empty name+address -> clean is empty string
[PASS] French '&' stripped, not swallowed
[PASS] French address token
[PASS] French 'R.' -> period stripped, not glued
[PASS] French accents stripped (é->e, ë->e)
[PASS] Real EDA hard-example records normalize without crashing
[PASS] 'AF-0684' -> {'af684'}
[PASS] '1056-1060' -> {'1056','1060'}
[PASS] Documented limitation: '48' vs '4-8' disjoint (expected)
[PASS] France 'sarl' derived into stoplist purely from DF
[PASS] US 'llc' derived into stoplist purely from DF
[PASS] Partitions don't leak (France stoplist has no 'llc')
[PASS] core_tokens drops high-DF 'sarl', keeps 'company7'
[PASS] No hardcoded country-name branching

All checks passed.
